# Explainability-Guided SegFormer-B0 — Full-Scale Training (Colab)

Scales up `train_segformer_smoke.py` / `eval_segformer.py` / `generate_attention_figures.py` (verified locally on CPU, ~460 images, 8 epochs — see `results/train_summary_*.json`) to the full dataset and epoch count from the proposal (§4.1: 5,000 images, 70/15/15 split) on a GPU.

Every piece run here — the SegFormer-B0 model, attention-extraction hooks, adapted Grad-Rollout, and the Attention Consistency Loss (incl. the double-backprop training step) — is the exact same code from `attention_consistency/`, unit-tested in `tests/`. Nothing is reimplemented in this notebook; it just runs the package at full scale.

## Step 0: Upload the package to Drive

Upload the whole `Phase1/Dinura-Person3/` folder (needs `attention_consistency/`, `train_segformer_smoke.py`, `eval_segformer.py`, `generate_attention_figures.py`) **and** `Phase1/Kalana-Person2/{images,masks}` (the dataset) to your Google Drive, e.g. under `MyDrive/DNN-Project/`. Adjust `DRIVE_BASE` below to match.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q transformers thop

import sys, os

DRIVE_BASE = '/content/drive/MyDrive/DNN-Project'  # ⬅️ adjust to your Drive path
PERSON3_DIR = os.path.join(DRIVE_BASE, 'Phase1', 'Dinura-Person3')
KALANA_IMAGES = os.path.join(DRIVE_BASE, 'Phase1', 'Kalana-Person2')

sys.path.insert(0, PERSON3_DIR)

# attention_consistency/data.py resolves the dataset relative to its own
# file location (../Kalana-Person2/{images,masks}); if your Drive layout
# differs, override the two module-level paths directly instead:
import attention_consistency.data as data_mod
from pathlib import Path
data_mod.IMG_DIR = Path(KALANA_IMAGES) / 'images'
data_mod.MASK_DIR = Path(KALANA_IMAGES) / 'masks'
print('images dir:', data_mod.IMG_DIR, '-> exists:', data_mod.IMG_DIR.exists())

## Step 1: Device check

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## Step 2: Full-scale training — both variants

Proposal-scale split: 5,000 images total, 70/15/15 → 3500/750/750. Increase `--epochs` from the CPU smoke run's 8 → 20 (proposal-scale) now that a GPU is available. This reuses `train_variant()` directly — same code path as the local run, just bigger numbers and (if CUDA is available) should be moved onto the GPU; the current `attention_consistency` modules run on whatever device the input tensors are on, so the only change needed for GPU is `.to(device)` on the model and each batch, added inline below rather than editing the shared module (keeps the CPU smoke-scale path untouched).

In [ ]:
import types
import train_segformer_smoke as T

device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Monkey-patch to_model_input / model placement onto `device` without
# touching the shared (CPU-verified) module files.
_orig_to_model_input = data_mod.to_model_input
def to_model_input_device(images):
    return _orig_to_model_input(images).to(device)
T.to_model_input = to_model_input_device
data_mod.to_model_input = to_model_input_device

_orig_build = T.build_segformer
def build_segformer_device(*a, **kw):
    return _orig_build(*a, **kw).to(device)
T.build_segformer = build_segformer_device

class Args:
    n_train, n_val, n_test = 3500, 750, 750
    epochs = 20
    batch_size = 16       # GPU can afford a larger vanilla-variant batch
    lr = 6e-5
    lambda2 = 0.3
    sigma = 8.0
    att_mode = 'mse'
    seed = 42

args = Args()
torch.manual_seed(args.seed)
for variant in ('vanilla', 'att'):
    T.train_variant(variant, args)

## Step 3: Evaluate both checkpoints (Dice / IoU / F1 / AAMO / efficiency)

Same held-out test split (`n_train, n_val, n_test, seed` must match Step 2). Reuses Person 4 (Lasana)'s `metrics.py` / `aamo.py` / `efficiency.py` exactly like the CPU run does, so numbers are directly comparable to the U-Net baseline row.

In [ ]:
import eval_segformer as E

class EvalArgs:
    n_train, n_val, n_test = 3500, 750, 750
    seed = 42

for variant in ('vanilla', 'att'):
    E.evaluate_variant(variant, EvalArgs())

## Step 4: Qualitative attention-drift figures

In [ ]:
!python generate_attention_figures.py --n 3

## Step 5: Save everything back to Drive

`results/` and `checkpoints/` already live under `PERSON3_DIR` on Drive (we `sys.path.insert`-ed straight into the Drive copy, so writes went there directly) — nothing further to copy. Download `results/attention_drift_figures/*.png` and `results/baseline_comparison.md` for the full paper.